In [1]:
# Libraries and configurations
import sys
from pathlib import Path
import json

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision import transforms
from torchinfo import summary

# Visualization and analysis
import os
import seaborn as sns
import PIL
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import cohen_kappa_score

# Adds the project root to the path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from config import (
    get_dataset_paths
)

from src.dataset import DataModuleParasite
from src import utils
from src import models
from src import trainer

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

### Eggs: Trained from Scratch

In [15]:
# Configuration for Eggs Dataset

CONFIG_EGG = {
    'dataset_name': 'eggs',
    'split': [1, 2, 3],
    'percentage': [1, 5, 25, 50, 75, 100],
    'num_classes': 9,
    'batch_size': 16,
    'num_workers': 4,
    'image_size': 200,
    'lr': 1e-4,
    'num_epochs': 100,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Create dataloaders
dataset_name = CONFIG_EGG['dataset_name']
model_name = 'squeezenet_scratch'
type_model = 'squeezenet'
pre_trained = False
path = f"{model_name}/{dataset_name}"
config = CONFIG_EGG
dataloaders = {}

transforms_egg = transforms.Compose([
    transforms.Resize((config['image_size'], config['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataloaders = utils.create_dataloaders(config=config, transforms=transforms_egg)

model_scratch, _, _, _ = models.create_model(type_model=type_model, description_path=f"{path}/{model_name}_model_params_{dataset_name}.txt", config=config, pre_trained=pre_trained)

n_flops, n_params = utils.count_flops(model_scratch, input_size=(1, 3, config['image_size'], config['image_size']))
print(f"Model: {model_name} - Dataset: {dataset_name}")
print(f"FLOPs: {n_flops} FLOPs")
print(f"Parameters: {n_params} ")

Model: squeezenet_scratch - Dataset: eggs
FLOPs: 214822361.0 FLOPs
Parameters: 727113.0 


In [16]:
# Training loop with pre-trained weights
historic_train = trainer.train_loop(config, dataloaders, path, dataset_name, model_name=model_name, type_model=type_model, pre_trained=pre_trained)

# Save training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

Early stopping at epoch 73 for split 1 and percentage 1%


Early stopping at epoch 64 for split 2 and percentage 1%


Early stopping at epoch 21 for split 3 and percentage 1%


Early stopping at epoch 22 for split 1 and percentage 5%


Early stopping at epoch 22 for split 2 and percentage 5%


...

Early stopping at epoch 64 for split 2 and percentage 75%


Early stopping at epoch 77 for split 3 and percentage 75%


Early stopping at epoch 72 for split 1 and percentage 100%


Early stopping at epoch 84 for split 2 and percentage 100%


Early stopping at epoch 78 for split 3 and percentage 100%


In [17]:
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

results_test = utils.evaluate_test_set(config, path, model_name, dataloaders, type_model=type_model, pre_trained=pre_trained)  

# Save results to JSON
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

utils.calculate_and_save_reports(config, path, model_name)

Aggregated classification reports saved to squeezenet_scratch/eggs/squeezenet_scratch_aggregated_classification_report_eggs.txt


### Larvae: Trained from Scratch

In [20]:
# Configuration for Larvae Dataset
CONFIG_LARVAE = {
    'dataset_name': 'larvae',
    'split': [1,2,3],
    'percentage': [1, 5, 25, 50, 75, 100],
    'num_classes': 2,
    'batch_size': 16,
    'num_workers': 4,
    'image_size': 200,
    'lr': 1e-4,
    'num_epochs': 100,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Create dataloaders
dataset_name = CONFIG_LARVAE['dataset_name']
model_name = 'squeezenet_scratch'
type_model = 'squeezenet'
pre_trained = False
path = f"{model_name}/{dataset_name}"
config = CONFIG_LARVAE
dataloaders = {}

transforms_larvae = transforms.Compose([
    transforms.Resize((config['image_size'], config['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataloaders = utils.create_dataloaders(config=config, transforms=transforms_larvae)

model_scratch, _, _, _ = models.create_model(type_model=type_model, description_path=f"{path}/{model_name}_model_params_{dataset_name}.txt", config=config, pre_trained=pre_trained)

n_flops, n_params = utils.count_flops(model_scratch, input_size=(1, 3, config['image_size'], config['image_size']))
print(f"Model: {model_name} - Dataset: {dataset_name}")
print(f"FLOPs: {n_flops} FLOPs")
print(f"Parameters: {n_params} ")

Model: squeezenet_scratch - Dataset: larvae
FLOPs: 214305250.0 FLOPs
Parameters: 723522.0 


In [21]:
# Training loop with pre-trained weights
historic_train = trainer.train_loop(config, dataloaders, path, dataset_name, model_name=model_name, type_model=type_model, pre_trained=pre_trained)

# Save training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)

Early stopping at epoch 50 for split 1 and percentage 1%


Early stopping at epoch 25 for split 2 and percentage 1%


Early stopping at epoch 21 for split 3 and percentage 1%


Early stopping at epoch 21 for split 1 and percentage 5%


Early stopping at epoch 21 for split 2 and percentage 5%


...

Early stopping at epoch 50 for split 2 and percentage 75%


Early stopping at epoch 49 for split 3 and percentage 75%


Early stopping at epoch 52 for split 1 and percentage 100%


Early stopping at epoch 56 for split 2 and percentage 100%


Early stopping at epoch 58 for split 3 and percentage 100%


In [22]:
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

results_test = utils.evaluate_test_set(config, path, model_name, dataloaders, type_model=type_model, pre_trained=pre_trained)  

# Save results to JSON
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

utils.calculate_and_save_reports(config, path, model_name)

Aggregated classification reports saved to squeezenet_scratch/larvae/squeezenet_scratch_aggregated_classification_report_larvae.txt


### Cysts: Trained from Scratch

In [23]:
# Configuration for Cisto Dataset
CONFIG_CISTO = {
    'dataset_name': 'cistos',
    'split': [1, 2, 3],
    'percentage': [1, 5, 25, 50, 75, 100],
    'num_classes': 7,
    'batch_size': 16,
    'num_workers': 4,
    'image_size': 200,
    'lr': 1e-4,
    'num_epochs': 100,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Create dataloaders
dataset_name = CONFIG_CISTO['dataset_name']
model_name = 'squeezenet_scratch'
type_model = 'squeezenet'
pre_trained = False
path = f"{model_name}/{dataset_name}"
config = CONFIG_CISTO
dataloaders = {}

transforms_cistos = transforms.Compose([
    transforms.Resize((config['image_size'], config['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataloaders = utils.create_dataloaders(config=config, transforms=transforms_cistos)

model_scratch, _, _, _ = models.create_model(type_model=type_model, description_path=f"{path}/{model_name}_model_params_{dataset_name}.txt", config=config, pre_trained=pre_trained)

n_flops, n_params = utils.count_flops(model_scratch, input_size=(1, 3, config['image_size'], config['image_size']))
print(f"Model: {model_name} - Dataset: {dataset_name}")
print(f"FLOPs: {n_flops} FLOPs")
print(f"Parameters: {n_params} ")

Model: squeezenet_scratch - Dataset: cistos
FLOPs: 214674615.0 FLOPs
Parameters: 726087.0 


In [24]:
# Training loop with pre-trained weights
historic_train = trainer.train_loop(config, dataloaders, path, dataset_name, model_name=model_name, type_model=type_model, pre_trained=pre_trained)

# Save training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)

Early stopping at epoch 69 for split 1 and percentage 1%


Early stopping at epoch 83 for split 2 and percentage 1%


Early stopping at epoch 67 for split 3 and percentage 1%


Early stopping at epoch 21 for split 1 and percentage 5%


Early stopping at epoch 22 for split 2 and percentage 5%


...

Early stopping at epoch 82 for split 2 and percentage 75%


Early stopping at epoch 61 for split 3 and percentage 75%


Early stopping at epoch 70 for split 1 and percentage 100%


Early stopping at epoch 71 for split 2 and percentage 100%


Early stopping at epoch 90 for split 3 and percentage 100%


In [25]:
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

results_test = utils.evaluate_test_set(config, path, model_name, dataloaders, type_model=type_model, pre_trained=pre_trained)  

# Save results to JSON
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

utils.calculate_and_save_reports(config, path, model_name)

Aggregated classification reports saved to squeezenet_scratch/cistos/squeezenet_scratch_aggregated_classification_report_cistos.txt


### Eggs: Pre-trained on ImageNet

In [26]:
# Configuration for Eggs Dataset

CONFIG_EGG = {
    'dataset_name': 'eggs',
    'split': [1, 2, 3],
    'percentage': [1, 5, 25, 50, 75, 100],
    'num_classes': 9,
    'batch_size': 16,
    'num_workers': 4,
    'image_size': 200,
    'lr': 1e-4,
    'num_epochs': 100,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Create dataloaders
dataset_name = CONFIG_EGG['dataset_name']
model_name = 'squeezenet_pretrained'
type_model = 'squeezenet'
pre_trained = True
path = f"{model_name}/{dataset_name}"
config = CONFIG_EGG
dataloaders = {}

transforms_egg = transforms.Compose([
    transforms.Resize((config['image_size'], config['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataloaders = utils.create_dataloaders(config=config, transforms=transforms_egg)

model_scratch, _, _, _ = models.create_model(type_model=type_model, description_path=f"{path}/{model_name}_model_params_{dataset_name}.txt", config=config, pre_trained=pre_trained)

n_flops, n_params = utils.count_flops(model_scratch, input_size=(1, 3, config['image_size'], config['image_size']))
print(f"Model: {model_name} - Dataset: {dataset_name}")
print(f"FLOPs: {n_flops} FLOPs")
print(f"Parameters: {n_params} ")

Model: squeezenet_pretrained - Dataset: eggs
FLOPs: 214822361.0 FLOPs
Parameters: 727113.0 


In [27]:
# Training loop with pre-trained weights
historic_train = trainer.train_loop(config, dataloaders, path, dataset_name, model_name=model_name, type_model=type_model, pre_trained=pre_trained)

# Save training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

Split 1 - Percentage 1%:   0%|          | 0/100 [00:00<?, ?it/s]

Early stopping at epoch 44 for split 1 and percentage 1%


Early stopping at epoch 72 for split 3 and percentage 1%


Early stopping at epoch 54 for split 1 and percentage 5%


Early stopping at epoch 57 for split 2 and percentage 5%


...

Early stopping at epoch 47 for split 2 and percentage 75%


Early stopping at epoch 37 for split 3 and percentage 75%


Early stopping at epoch 39 for split 1 and percentage 100%


Early stopping at epoch 40 for split 2 and percentage 100%


Early stopping at epoch 38 for split 3 and percentage 100%


In [28]:
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

results_test = utils.evaluate_test_set(config, path, model_name, dataloaders, type_model=type_model, pre_trained=pre_trained)  

# Save results to JSON
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

utils.calculate_and_save_reports(config, path, model_name)

Aggregated classification reports saved to squeezenet_pretrained/eggs/squeezenet_pretrained_aggregated_classification_report_eggs.txt


### Larvae: Pre-trained on ImageNet

In [29]:
# Configuration for Larvae Dataset
CONFIG_LARVAE = {
    'dataset_name': 'larvae',
    'split': [1,2,3],
    'percentage': [1, 5, 25, 50, 75, 100],
    'num_classes': 2,
    'batch_size': 16,
    'num_workers': 4,
    'image_size': 200,
    'lr': 1e-4,
    'num_epochs': 100,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Create dataloaders
dataset_name = CONFIG_LARVAE['dataset_name']
model_name = 'squeezenet_pretrained'
type_model = 'squeezenet'
pre_trained = True
path = f"{model_name}/{dataset_name}"
config = CONFIG_LARVAE
dataloaders = {}

transforms_larvae = transforms.Compose([
    transforms.Resize((config['image_size'], config['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataloaders = utils.create_dataloaders(config=config, transforms=transforms_larvae)

model_scratch, _, _, _ = models.create_model(type_model=type_model, description_path=f"{path}/{model_name}_model_params_{dataset_name}.txt", config=config, pre_trained=pre_trained)

n_flops, n_params = utils.count_flops(model_scratch, input_size=(1, 3, config['image_size'], config['image_size']))
print(f"Model: {model_name} - Dataset: {dataset_name}")
print(f"FLOPs: {n_flops} FLOPs")
print(f"Parameters: {n_params} ")

Model: squeezenet_pretrained - Dataset: larvae
FLOPs: 214305250.0 FLOPs
Parameters: 723522.0 


In [30]:
# Training loop with pre-trained weights
historic_train = trainer.train_loop(config, dataloaders, path, dataset_name, model_name=model_name, type_model=type_model, pre_trained=pre_trained)

# Save training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

Early stopping at epoch 21 for split 1 and percentage 1%


Early stopping at epoch 24 for split 2 and percentage 1%


Early stopping at epoch 42 for split 3 and percentage 1%


Early stopping at epoch 27 for split 1 and percentage 5%


Early stopping at epoch 36 for split 2 and percentage 5%


...

Early stopping at epoch 31 for split 2 and percentage 75%


Early stopping at epoch 30 for split 3 and percentage 75%


Early stopping at epoch 32 for split 1 and percentage 100%


Early stopping at epoch 32 for split 2 and percentage 100%


Early stopping at epoch 29 for split 3 and percentage 100%


In [ ]:
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

results_test = utils.evaluate_test_set(config, path, model_name, dataloaders, type_model=type_model, pre_trained=pre_trained)  

# Save results to JSON
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

utils.calculate_and_save_reports(config, path, model_name)

### Cysts: Pre-trained on ImageNet

In [31]:
# Configuration for Cisto Dataset
CONFIG_CISTO = {
    'dataset_name': 'cistos',
    'split': [1, 2, 3],
    'percentage': [1, 5, 25, 50, 75, 100],
    'num_classes': 7,
    'batch_size': 16,
    'num_workers': 4,
    'image_size': 200,
    'lr': 1e-4,
    'num_epochs': 100,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Create dataloaders
dataset_name = CONFIG_CISTO['dataset_name']
model_name = 'squeezenet_pretrained'
type_model = 'squeezenet'
pre_trained = True
path = f"{model_name}/{dataset_name}"
config = CONFIG_CISTO
dataloaders = {}

transforms_cistos = transforms.Compose([
    transforms.Resize((config['image_size'], config['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataloaders = utils.create_dataloaders(config=config, transforms=transforms_cistos)

model_scratch, _, _, _ = models.create_model(type_model=type_model, description_path=f"{path}/{model_name}_model_params_{dataset_name}.txt", config=config, pre_trained=pre_trained)

n_flops, n_params = utils.count_flops(model_scratch, input_size=(1, 3, config['image_size'], config['image_size']))
print(f"Model: {model_name} - Dataset: {dataset_name}")
print(f"FLOPs: {n_flops} FLOPs")
print(f"Parameters: {n_params} ")

Model: squeezenet_pretrained - Dataset: cistos
FLOPs: 214674615.0 FLOPs
Parameters: 726087.0 


In [32]:
# Training loop with pre-trained weights
historic_train = trainer.train_loop(config, dataloaders, path, dataset_name, model_name=model_name, type_model=type_model, pre_trained=pre_trained)

# Save training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

Early stopping at epoch 21 for split 1 and percentage 1%


Early stopping at epoch 29 for split 2 and percentage 1%


Early stopping at epoch 68 for split 3 and percentage 1%


Early stopping at epoch 56 for split 1 and percentage 5%


Early stopping at epoch 59 for split 2 and percentage 5%


...

Early stopping at epoch 52 for split 2 and percentage 75%


Early stopping at epoch 56 for split 3 and percentage 75%


Early stopping at epoch 49 for split 1 and percentage 100%


Early stopping at epoch 50 for split 2 and percentage 100%


Early stopping at epoch 46 for split 3 and percentage 100%


In [33]:
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

results_test = utils.evaluate_test_set(config, path, model_name, dataloaders, type_model=type_model, pre_trained=pre_trained)  

# Save results to JSON
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

utils.calculate_and_save_reports(config, path, model_name)

Aggregated classification reports saved to squeezenet_pretrained/cistos/squeezenet_pretrained_aggregated_classification_report_cistos.txt
